# Chapter 3 — 文本预处理

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaosfrey-arch/news-classification-tutorial/blob/main/chapter03_preprocessing.ipynb)

**本章目标：** 把原始文本清洗干净，切分成训练集和测试集。

**预计时间：** 45 分钟

## 3.1 为什么要预处理？

原始文本充满了「噪声」——对分类没用、甚至有干扰的内容：

- `"Apple"` 和 `"apple"` 是同一个词，但计算机认为不同
- `"https://www.bbc.com/news/..."` 这段 URL 不携带任何分类信息
- `","` `"."` `"!"` 标点符号对判断新闻类别没帮助
- `"123"` `"2024"` 数字通常也不重要

**目标：** 把文章变成一串干净的小写英文单词，去掉所有噪声。

## 3.2 正则表达式（Regular Expression）简介

正则表达式是一种「模式描述语言」，用来在文本里查找/替换特定的字符组合。

不用深入学，只需理解这个作业里用到的几个符号：

| 符号 | 含义 | 例子 |
|------|------|------|
| `[abc]` | 匹配 a 或 b 或 c | |
| `[^abc]` | 匹配**不是** a、b、c 的任何字符 | |
| `[a-z]` | 匹配 a 到 z 的任意小写字母 | |
| `\s` | 匹配空白字符（空格、换行等）| |
| `\S` | 匹配非空白字符 | |
| `+` | 前面的模式出现 1 次或多次 | |
| `http\S+` | 匹配 http 开头后接任意非空白字符 | 匹配 URL |

In [ ]:
import re  # re 是 Python 内置的正则表达式库

# 示例
text = "Visit https://bbc.com for more! It's FREE in 2024."

# 1. 转小写
step1 = text.lower()
print('小写：', step1)

# 2. 去掉 URL（http 或 www 开头的字符串）
step2 = re.sub(r'http\S+|www\S+|https\S+', '', step1)
print('去URL：', step2)

# 3. 只保留字母和空格，其他全换成空格
# [^a-z\s] = 不是小写字母也不是空白的字符
step3 = re.sub(r'[^a-z\s]', ' ', step2)
print('去标点：', step3)

# 4. 合并多个空格为一个，去掉首尾空格
step4 = re.sub(r'\s+', ' ', step3).strip()
print('清理后：', step4)

## 3.3 把四步合并成一个函数

In [ ]:
def clean_text(text):
    """
    清洗文本：转小写 → 去 URL → 去非字母字符 → 合并空格
    参数：text (str) - 原始文本
    返回：cleaned (str) - 清洗后的文本
    """
    text = str(text)                              # 确保是字符串（处理 NaN）
    text = text.lower()                           # 全部转小写
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)  # 去 URL
    text = re.sub(r'[^a-z\s]', ' ', text)        # 只保留字母和空格
    text = re.sub(r'\s+', ' ', text).strip()      # 合并多余空格
    return text

# 测试函数
examples = [
    "Apple Inc. reported $100B profit in Q3 2024!",
    "Visit https://reuters.com for breaking news.",
    "NASA's Mars rover discovers ancient rock formations."
]

for ex in examples:
    print('原始：', ex)
    print('清洗：', clean_text(ex))
    print()

## 3.4 处理整个数据集

In [ ]:
import pandas as pd

df = pd.read_csv('dataset.csv')
print(f'原始数据：{len(df):,} 行')
print(f'缺失值：\n{df.isnull().sum()}')

In [ ]:
# 步骤 1：删除 Class 列缺失的行（没有标签，无法用于训练）
df_clean = df.dropna(subset=['Class']).copy()
print(f'删除缺失标签后：{len(df_clean):,} 行（删了 {len(df)-len(df_clean)} 行）')

# 步骤 2：Description 缺失的行保留，用空字符串填充
# 原因：Title 还在，仍然有部分信息
df_clean['Description'] = df_clean['Description'].fillna('')

# 步骤 3：合并 Title + Description 为单一文本列
# 原因：标题是高密度信息，描述是详细背景，两者互补
df_clean['text'] = df_clean['Title'] + ' ' + df_clean['Description']

# 步骤 4：应用清洗函数（这步可能需要 1-2 分钟）
print('正在清洗文本，请稍候...')
df_clean['text'] = df_clean['text'].apply(clean_text)
print('清洗完成！')

# 预览
print('\n原始 Title：', df.iloc[0]['Title'])
print('清洗后 text：', df_clean.iloc[0]['text'][:100], '...')

## 3.5 标签编码（Label Encoding）

机器学习模型处理数字比处理文字更容易。我们需要把类别名转成数字：

| 类别 | 编码 |
|------|------|
| Business | 0 |
| Science/Tech | 1 |
| Sports | 2 |
| World | 3 |

In [ ]:
from sklearn.preprocessing import LabelEncoder

# LabelEncoder 自动把字符串标签转成 0,1,2,3...
le = LabelEncoder()
df_clean['label'] = le.fit_transform(df_clean['Class'])

print('标签映射：')
for cls, code in zip(le.classes_, le.transform(le.classes_)):
    print(f'  {cls} → {code}')

print('\n编码后的前 5 行：')
print(df_clean[['Class', 'label']].head())

## 3.6 划分训练集和测试集

**核心原则：** 测试集在训练时必须完全不可见，否则就是作弊。

类比：**平时练习题（训练集）** vs **期末考试（测试集）**。如果考试题提前让你看到，那成绩没有意义。

**为什么是 70/30？**
- 训练集越大，模型学得越好
- 测试集太小，评估结果不可靠
- 70/30 是文本分类任务的常用比例

**什么是「分层」（Stratified）划分？**

如果你随机切，可能碰巧某个类别在测试集里特别多。分层切分保证每个类别在训练集和测试集里的比例相同。

In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean['text'].values   # 特征：文章文本
y = df_clean['label'].values  # 标签：0/1/2/3

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,    # 30% 作为测试集
    random_state=42,   # 固定随机种子，保证每次运行结果一致
    stratify=y         # 分层划分：保持每类比例
)

print(f'训练集：{len(X_train):,} 篇文章')
print(f'测试集：{len(X_test):,} 篇文章')

# 验证分层是否成功
import numpy as np
for name, y_split in [('训练集', y_train), ('测试集', y_test)]:
    unique, counts = np.unique(y_split, return_counts=True)
    print(f'\n{name}各类别数量：')
    for cls_id, cnt in zip(unique, counts):
        print(f'  {le.classes_[cls_id]}: {cnt:,} ({cnt/len(y_split)*100:.1f}%)')

## 🏋️ 小练习

In [ ]:
# 练习 1：改变 random_state（比如改成 0 或 100），
# 观察训练集/测试集的大小是否变化（应该不变），
# 但具体哪篇文章在哪个集合会变

# 你的代码：


In [ ]:
# 练习 2：用 clean_text 函数清洗下面这段文字，打印结果
test_sentence = "BREAKING: Tesla's stock rose 15% to $250.50 after CEO Elon Musk tweeted! Visit https://tesla.com"

# 你的代码：


## 总结

**预处理流程：**
```
原始 CSV
  → 删除 Class 缺失的行（-500 行）
  → Description 缺失填空字符串
  → 合并 Title + Description
  → clean_text()（小写、去URL、去标点）
  → LabelEncoder（类别名 → 数字）
  → train_test_split（70% 训练 / 30% 测试，分层）
```

**下一章 →** Chapter 4：模型是什么？训练是什么？